## Imports and Config

In [12]:
import pandas as pd
import numpy as np
import random
import os
from tqdm import tqdm
np.random.seed(42)
random.seed(42)

DATA_PATH = "/app/data/raw"
os.makedirs(DATA_PATH, exist_ok=True)

## Build Route Dimension

In [2]:
cities = ["Bangalore", "Chennai", "Hyderabad", "Mumbai", "Pune", "Delhi"]

In [4]:
distance_map = {
    ("Bangalore", "Chennai"): 350,
    ("Bangalore", "Hyderabad"): 570,
    ("Bangalore", "Mumbai"): 980,
    ("Bangalore", "Pune"): 840,
    ("Bangalore", "Delhi"): 2150,

    ("Chennai", "Hyderabad"): 630,
    ("Chennai", "Mumbai"): 1330,
    ("Chennai", "Pune"): 1150,
    ("Chennai", "Delhi"): 2200,

    ("Hyderabad", "Mumbai"): 710,
    ("Hyderabad", "Pune"): 560,
    ("Hyderabad", "Delhi"): 1550,

    ("Mumbai", "Pune"): 150,
    ("Mumbai", "Delhi"): 1400,

    ("Pune", "Delhi"): 1450,
}

In [5]:
def get_distance(src, dest):
    if (src, dest) in distance_map:
        base = distance_map[(src, dest)]
    elif (dest, src) in distance_map:
        base = distance_map[(dest, src)]
    else:
        base = random.randint(300, 1500)  # fallback (rare)

    # add slight randomness (real-world variation)
    return int(base * random.uniform(0.9, 1.1))

In [6]:
routes = []
route_id = 1

for src in cities:
    for dest in cities:
        if src != dest:
            routes.append({
                "route_id": route_id,
                "source_city": src,
                "destination_city": dest,
                "distance_km": get_distance(src, dest),
                "avg_speed_kmph": random.randint(40, 80),
                "traffic_factor": round(random.uniform(0.8, 1.5), 2)
            })
            route_id += 1

routes_df = pd.DataFrame(routes)
routes_df.head()

,route_id,source_city,destination_city,distance_km,avg_speed_kmph,traffic_factor
0,1,Bangalore,Chennai,342,69,0.90
1,2,Bangalore,Hyderabad,528,75,1.18
2,3,Bangalore,Mumbai,1028,67,1.43
3,4,Bangalore,Pune,823,54,1.50
4,5,Bangalore,Delhi,1994,71,0.86


## Trucks Dimension

In [7]:
truck_models = [
    {"brand": "BharatBenz", "model": "1217R", "capacity_range": (10, 16), "efficiency_range": (4, 6)},
    {"brand": "Tata", "model": "Ultra 1518", "capacity_range": (12, 18), "efficiency_range": (4, 7)},
    {"brand": "Ashok Leyland", "model": "Ecomet", "capacity_range": (10, 16), "efficiency_range": (4, 6)},
    {"brand": "Volvo", "model": "FM 420", "capacity_range": (20, 30), "efficiency_range": (3, 5)},
    {"brand": "MAN", "model": "CLA 31.300", "capacity_range": (18, 28), "efficiency_range": (3, 5)},
    {"brand": "Daimler", "model": "Actros", "capacity_range": (20, 30), "efficiency_range": (3, 5)},
]

In [9]:
n_trucks = 1000

truck_records = []

for i in range(1, n_trucks + 1):
    model_info = random.choice(truck_models)
    
    capacity = random.randint(*model_info["capacity_range"])
    efficiency = round(random.uniform(*model_info["efficiency_range"]), 2)

    truck_records.append({
        "truck_id": i,
        "brand": model_info["brand"],
        "model": model_info["model"],
        "capacity_tons": capacity,
        "fuel_efficiency_kmpl": efficiency,
        "status": random.choices(["available", "unavailable"], weights=[0.9, 0.1])[0]
    })

trucks_df = pd.DataFrame(truck_records)
trucks_df["cost_per_km"] = round(1 / trucks_df["fuel_efficiency_kmpl"], 3)
trucks_df.head()

,truck_id,brand,model,capacity_tons,fuel_efficiency_kmpl,status,cost_per_km
0,1,Volvo,FM 420,27,3.09,available,0.324
1,2,Ashok Leyland,Ecomet,10,5.41,available,0.185
2,3,Daimler,Actros,29,4.45,available,0.225
3,4,Daimler,Actros,25,4.04,available,0.248
4,5,MAN,CLA 31.300,19,3.87,available,0.258


## Drivers Dimension

In [10]:
n_drivers = 1000

drivers_df = pd.DataFrame({
    "driver_id": range(1, n_drivers + 1),
    "max_hours_per_day": np.random.randint(8, 12, n_drivers),
    "experience_level": np.random.choice(["junior", "mid", "senior"], n_drivers),
    "fatigue_factor": np.round(np.random.uniform(0.8, 1.2, n_drivers), 2)
})

drivers_df.head()

,driver_id,max_hours_per_day,experience_level,fatigue_factor
0,1,10,mid,1.16
1,2,11,senior,1.20
2,3,8,junior,0.82
3,4,10,junior,1.15
4,5,10,junior,1.01


## Trips Fact

In [11]:
n_trips = 100000

sampled_routes = routes_df.sample(n=n_trips, replace=True).reset_index(drop=True)

trips_df = pd.DataFrame({
    "trip_id": range(1, n_trips + 1),
    "route_id": sampled_routes["route_id"],
    "source_city": sampled_routes["source_city"],
    "destination_city": sampled_routes["destination_city"],
    "distance_km": sampled_routes["distance_km"],
    "load_tons": np.random.randint(1, 25, n_trips),
    "delivery_deadline_hours": np.random.randint(5, 48, n_trips)
})

trips_df.head()

,trip_id,route_id,source_city,destination_city,distance_km,load_tons,delivery_deadline_hours
0,1,20,Mumbai,Delhi,1506,2,31
1,2,2,Bangalore,Hyderabad,528,19,21
2,3,26,Delhi,Bangalore,2116,24,16
3,4,30,Delhi,Pune,1323,16,16
4,5,30,Delhi,Pune,1323,5,39


In [12]:
routes_df.to_parquet(f"{DATA_PATH}/routes.parquet", index=False)
trucks_df.to_parquet(f"{DATA_PATH}/trucks.parquet", index=False)
drivers_df.to_parquet(f"{DATA_PATH}/drivers.parquet", index=False)
trips_df.to_parquet(f"{DATA_PATH}/trips.parquet", index=False)

In [13]:
print("Routes:", routes_df.shape)
print("Trucks:", trucks_df.shape)
print("Drivers:", drivers_df.shape)
print("Trips:", trips_df.shape)

Routes: (30, 6)
Trucks: (1000, 7)
Drivers: (1000, 4)
Trips: (100000, 7)


### Add Time Constraint

In [3]:
routes_df = pd.read_parquet(f"{DATA_PATH}/routes.parquet")
trucks_df = pd.read_parquet(f"{DATA_PATH}/trucks.parquet")
drivers_df = pd.read_parquet(f"{DATA_PATH}/drivers.parquet")
trips_df = pd.read_parquet(f"{DATA_PATH}/trips.parquet")

In [4]:
# Define date range (simulate 7 days)
date_range = pd.date_range(start="2026-01-01", periods=7)

trips_df["trip_date"] = np.random.choice(date_range, size=len(trips_df))
trips_df["planned_start_hour"] = np.random.randint(0, 24, len(trips_df))

In [5]:
# Merge route info
trips_df = trips_df.merge(
    routes_df[["route_id", "avg_speed_kmph", "traffic_factor"]],
    on="route_id",
    how="left"
)

# Calculate duration
trips_df["estimated_duration_hours"] = (
    trips_df["distance_km"] /
    (trips_df["avg_speed_kmph"] / trips_df["traffic_factor"])
).round(2)
trips_df = trips_df.drop(columns=["avg_speed_kmph", "traffic_factor"])


In [6]:
buffer_factor = np.random.uniform(1.1, 1.5, len(trips_df))

trips_df["delivery_deadline_hours"] = (
    trips_df["estimated_duration_hours"] * buffer_factor
).round(2)

In [7]:
# 10% trips will have tight/unrealistic deadlines
mask = np.random.rand(len(trips_df)) < 0.1

trips_df.loc[mask, "delivery_deadline_hours"] = (
    trips_df.loc[mask, "estimated_duration_hours"] *
    np.random.uniform(0.7, 0.95, mask.sum())
).round(2)

In [8]:
trips_df["is_deadline_feasible"] = (
    trips_df["estimated_duration_hours"] <= trips_df["delivery_deadline_hours"]
)
trips_df["trip_end_hour"] = (trips_df["planned_start_hour"] + trips_df["estimated_duration_hours"])

## Assignment System

In [9]:
available_trucks = trucks_df[trucks_df["status"] == "available"].copy()
available_trucks = available_trucks.sort_values("fuel_efficiency_kmpl",ascending = False)
driver_schedule = {
    driver_id: {} for driver_id in drivers_df["driver_id"]
}

In [13]:
assignments = []
assignment_id = 1

trips_df_sample = trips_df.sample(10000)

for date in tqdm(trips_df_sample["trip_date"].sort_values().unique(), desc="Days"):
    
    daily_trips = trips_df_sample[
        trips_df_sample["trip_date"] == date
    ]
    
    print(f"\nProcessing date: {date}, Trips: {len(daily_trips)}")
    for _, trip in tqdm(daily_trips.iterrows(), total=len(daily_trips), leave=False):
        
        eligible_drivers = drivers_df.sample(min(50, len(drivers_df))) 
        trip_start = trip["planned_start_hour"]
        trip_end = trip["trip_end_hour"]
        trip_date = trip["trip_date"]
        load = trip["load_tons"]
        
        # 1. Find valid trucks
        valid_trucks = available_trucks[
            available_trucks["capacity_tons"] >= load
        ]
        
        if valid_trucks.empty:
            continue
        
        # Pick cheapest truck (fuel efficiency proxy)
        truck = valid_trucks.iloc[0]
        
        # 2. Find valid driver
        assigned_driver = None
        
        for _, driver in eligible_drivers.iterrows():
            driver_id = driver["driver_id"]
            max_hours = driver["max_hours_per_day"]
            
            # Get existing schedule
            day_schedule = driver_schedule[driver_id].get(trip_date, [])
            
            # Check overlap
            overlap = any(
                not (trip_end <= s or trip_start >= e)
                for s, e in day_schedule
            )
            
            total_hours = sum(e - s for s, e in day_schedule)
            
            if not overlap and (total_hours + (trip_end - trip_start)) <= max_hours:
                assigned_driver = driver_id
                
                # Update schedule
                driver_schedule[driver_id].setdefault(trip_date, []).append(
                    (trip_start, trip_end)
                )
                break
        
        if assigned_driver is None:
            continue
        
        # 3. Create assignment
        assignments.append({
            "assignment_id": assignment_id,
            "trip_id": trip["trip_id"],
            "truck_id": truck["truck_id"],
            "driver_id": assigned_driver,
            "route_id": trip["route_id"],
            "trip_date": trip_date,
            "start_hour": trip_start,
            "end_hour": trip_end,
            "estimated_duration_hours": trip["estimated_duration_hours"]
        })
        
        assignment_id += 1


Days:   0%|          | 0/7 [00:00<?, ?it/s]


Processing date: 2026-01-01 00:00:00, Trips: 1419




  0%|          | 0/1419 [00:00<?, ?it/s]

  2%|▏         | 33/1419 [00:00<00:04, 325.99it/s]

  5%|▍         | 66/1419 [00:00<00:04, 316.78it/s]

  9%|▉         | 125/1419 [00:00<00:02, 438.13it/s]

 12%|█▏        | 170/1419 [00:00<00:03, 383.69it/s]

 15%|█▍        | 210/1419 [00:00<00:03, 363.10it/s]

 18%|█▊        | 253/1419 [00:00<00:03, 381.46it/s]

 21%|██        | 292/1419 [00:00<00:03, 334.18it/s]

 24%|██▎       | 334/1419 [00:00<00:03, 355.13it/s]

 26%|██▌       | 371/1419 [00:01<00:02, 352.02it/s]

 29%|██▉       | 413/1419 [00:01<00:02, 370.11it/s]

 32%|███▏      | 458/1419 [00:01<00:02, 387.21it/s]

 35%|███▌      | 498/1419 [00:01<00:02, 335.75it/s]

 39%|███▊      | 547/1419 [00:01<00:02, 371.18it/s]

 41%|████▏     | 586/1419 [00:01<00:02, 285.38it/s]

 44%|████▍     | 624/1419 [00:01<00:02, 304.17it/s]

 48%|████▊     | 679/1419 [00:01<00:02, 363.32it/s]

 51%|█████     | 719/1419 [00:02<00:01, 366.26it/s]

 54%|█████▍    | 767/1419 [00:02<00:01, 395.49it/s]

 57%


Processing date: 2026-01-02 00:00:00, Trips: 1426




  0%|          | 0/1426 [00:00<?, ?it/s]

  5%|▍         | 66/1426 [00:00<00:02, 648.08it/s]

  9%|▉         | 131/1426 [00:00<00:02, 585.12it/s]

 13%|█▎        | 190/1426 [00:00<00:02, 564.41it/s]

 17%|█▋        | 247/1426 [00:00<00:02, 509.61it/s]

 21%|██        | 299/1426 [00:00<00:02, 503.05it/s]

 25%|██▍       | 355/1426 [00:00<00:02, 518.33it/s]

 29%|██▉       | 415/1426 [00:00<00:01, 538.98it/s]

 34%|███▎      | 478/1426 [00:00<00:01, 564.35it/s]

 38%|███▊      | 539/1426 [00:00<00:01, 576.60it/s]

 42%|████▏     | 597/1426 [00:01<00:01, 567.33it/s]

 46%|████▌     | 654/1426 [00:01<00:01, 495.10it/s]

 50%|████▉     | 706/1426 [00:01<00:01, 483.40it/s]

 53%|█████▎    | 756/1426 [00:01<00:01, 456.64it/s]

 56%|█████▋    | 803/1426 [00:01<00:01, 432.55it/s]

 59%|█████▉    | 847/1426 [00:01<00:01, 361.58it/s]

 62%|██████▏   | 886/1426 [00:01<00:01, 306.44it/s]

 66%|██████▌   | 936/1426 [00:02<00:01, 349.01it/s]

 69%|██████▉   | 990/1426 [00:02<00:01, 394.33it/s]

 73

...



  0%|          | 0/1383 [00:00<?, ?it/s]

  4%|▍         | 55/1383 [00:00<00:02, 549.40it/s]

  8%|▊         | 114/1383 [00:00<00:02, 566.78it/s]

 12%|█▏        | 171/1383 [00:00<00:02, 516.09it/s]

 16%|█▌        | 224/1383 [00:00<00:02, 511.55it/s]

 20%|█▉        | 276/1383 [00:00<00:02, 498.96it/s]

 24%|██▎       | 328/1383 [00:00<00:02, 503.74it/s]

 28%|██▊       | 384/1383 [00:00<00:01, 519.48it/s]

 32%|███▏      | 437/1383 [00:00<00:01, 498.19it/s]

 35%|███▌      | 488/1383 [00:00<00:01, 450.55it/s]

 39%|███▊      | 534/1383 [00:01<00:01, 432.71it/s]

 42%|████▏     | 587/1383 [00:01<00:01, 456.54it/s]

 47%|████▋     | 645/1383 [00:01<00:01, 489.49it/s]

 50%|█████     | 695/1383 [00:01<00:01, 481.20it/s]

 54%|█████▍    | 749/1383 [00:01<00:01, 495.17it/s]

 58%|█████▊    | 808/1383 [00:01<00:01, 520.28it/s]

 62%|██████▏   | 861/1383 [00:01<00:01, 515.01it/s]

 67%|██████▋   | 920/1383 [00:01<00:00, 536.32it/s]

 70%|███████   | 974/1383 [00:01<00:00, 500.74it/s]

 74


Processing date: 2026-01-06 00:00:00, Trips: 1462




  0%|          | 0/1462 [00:00<?, ?it/s]

  3%|▎         | 39/1462 [00:00<00:03, 366.19it/s]

  5%|▌         | 76/1462 [00:00<00:04, 326.33it/s]

  7%|▋         | 109/1462 [00:00<00:05, 260.88it/s]

 10%|█         | 147/1462 [00:00<00:04, 297.63it/s]

 13%|█▎        | 189/1462 [00:00<00:03, 335.25it/s]

 15%|█▌        | 224/1462 [00:00<00:03, 327.73it/s]

 18%|█▊        | 265/1462 [00:00<00:03, 351.78it/s]

 22%|██▏       | 319/1462 [00:00<00:02, 406.73it/s]

 25%|██▍       | 365/1462 [00:01<00:02, 418.36it/s]

 28%|██▊       | 412/1462 [00:01<00:02, 431.99it/s]

 31%|███▏      | 457/1462 [00:01<00:02, 436.39it/s]

 34%|███▍      | 501/1462 [00:01<00:02, 427.81it/s]

 37%|███▋      | 545/1462 [00:01<00:02, 429.73it/s]

 40%|████      | 589/1462 [00:01<00:02, 356.96it/s]

 43%|████▎     | 627/1462 [00:01<00:02, 352.58it/s]

 46%|████▋     | 678/1462 [00:01<00:01, 392.29it/s]

 50%|████▉     | 725/1462 [00:01<00:01, 413.09it/s]

 53%|█████▎    | 768/1462 [00:02<00:01, 401.69it/s]

 55%


Processing date: 2026-01-07 00:00:00, Trips: 1429




  0%|          | 0/1429 [00:00<?, ?it/s]

  4%|▎         | 51/1429 [00:00<00:02, 508.31it/s]

  7%|▋         | 102/1429 [00:00<00:02, 475.87it/s]

 10%|█         | 150/1429 [00:00<00:03, 337.95it/s]

 13%|█▎        | 188/1429 [00:00<00:03, 350.48it/s]

 16%|█▌        | 226/1429 [00:00<00:03, 349.18it/s]

 20%|█▉        | 282/1429 [00:00<00:02, 411.81it/s]

 23%|██▎       | 332/1429 [00:00<00:02, 434.36it/s]

 27%|██▋       | 384/1429 [00:00<00:02, 458.22it/s]

 31%|███       | 440/1429 [00:01<00:02, 488.06it/s]

 35%|███▍      | 499/1429 [00:01<00:01, 517.71it/s]

 39%|███▉      | 555/1429 [00:01<00:01, 528.29it/s]

 43%|████▎     | 611/1429 [00:01<00:01, 537.37it/s]

 47%|████▋     | 666/1429 [00:01<00:01, 509.68it/s]

 50%|█████     | 718/1429 [00:01<00:02, 349.22it/s]

 53%|█████▎    | 764/1429 [00:01<00:01, 373.10it/s]

 57%|█████▋    | 816/1429 [00:01<00:01, 407.08it/s]

 60%|██████    | 862/1429 [00:02<00:01, 390.35it/s]

 64%|██████▍   | 919/1429 [00:02<00:01, 433.20it/s]

 68

## Upgrading Assignment Functions (Considering more parameters like Fuel Cost, Driver Cost, Penalty)

In [14]:
def compute_cost(truck, trip):
    """
    Compute total operational cost for assigning a truck to a trip.
    
    Inputs:
        truck -> row from trucks_df
        trip  -> row from trips_df
        
    Returns:
        total_cost (float)
    """
    
    # ---------------------------
    # CONFIG (can move to config file later)
    # ---------------------------
    FUEL_PRICE = 100      # per litre
    DRIVER_RATE = 200     # per hour
    LATE_PENALTY = 500    # per hour delay
    
    # ---------------------------
    # EXTRACT VALUES
    # ---------------------------
    distance = trip["distance_km"]
    duration = trip["estimated_duration_hours"]
    deadline = trip["delivery_deadline_hours"]
    load = trip["load_tons"]
    base_efficiency = truck["fuel_efficiency_kmpl"]
    
    # ---------------------------
    # REALISM ADJUSTMENTS
    # ---------------------------
    
    # Load impact → heavier load reduces efficiency
    load_factor = 1 + (load / truck["capacity_tons"]) * 0.2
    
    # Optional: traffic impact (if available in trip later)
    # traffic_factor = trip.get("traffic_factor", 1.0)
    
    adjusted_efficiency = base_efficiency / load_factor
    
    # Avoid divide-by-zero
    adjusted_efficiency = max(adjusted_efficiency, 0.1)
    
    # ---------------------------
    # COST COMPONENTS
    # ---------------------------
    
    # Fuel cost (distance-based)
    fuel_cost = (distance / adjusted_efficiency) * FUEL_PRICE
    
    # Driver cost (time-based)
    driver_cost = duration * DRIVER_RATE
    
    # Delay penalty
    delay = max(0, duration - deadline)
    penalty = delay * LATE_PENALTY
    
    # ---------------------------
    # TOTAL COST
    # ---------------------------
    total_cost = fuel_cost + driver_cost + penalty
    
    return round(total_cost, 2)

In [15]:
def assign_trips(trips_df, trucks_df, drivers_df, max_driver_candidates=50, max_truck_candidates=5):
    """
    Assign trucks and drivers to trips using cost-based heuristic.

    Parameters:
        trips_df (DataFrame)
        trucks_df (DataFrame)
        drivers_df (DataFrame)
        max_driver_candidates (int): number of drivers to sample per trip
        max_truck_candidates (int): number of trucks to evaluate per trip

    Returns:
        assignments_df (DataFrame)
    """
    
    from tqdm import tqdm
    
    # ---------------------------
    # PREP
    # ---------------------------
    
    assignments = []
    assignment_id = 1
    
    # Only available trucks
    available_trucks = trucks_df[trucks_df["status"] == "available"].copy()
    
    # Pre-sort trucks (better first)
    available_trucks = available_trucks.sort_values(
        "fuel_efficiency_kmpl", ascending=False
    )
    
    # Driver schedule tracker
    driver_schedule = {
        driver_id: {} for driver_id in drivers_df["driver_id"]
    }
    
    # ---------------------------
    # LOOP BY DAY (BATCH STYLE)
    # ---------------------------
    
    for date in tqdm(trips_df["trip_date"].sort_values().unique(), desc="Days"):
        
        daily_trips = trips_df[trips_df["trip_date"] == date]
        
        for _, trip in tqdm(daily_trips.iterrows(), total=len(daily_trips), leave=False):
            
            trip_start = trip["planned_start_hour"]
            trip_end = trip["trip_end_hour"]
            load = trip["load_tons"]
            
            # ---------------------------
            # FILTER VALID TRUCKS
            # ---------------------------
            valid_trucks = available_trucks[
                available_trucks["capacity_tons"] >= load
            ].head(max_truck_candidates)
            
            if valid_trucks.empty:
                continue
            
            # ---------------------------
            # SAMPLE DRIVER CANDIDATES
            # ---------------------------
            eligible_drivers = drivers_df.sample(
                min(max_driver_candidates, len(drivers_df))
            )
            
            # ---------------------------
            # FIND BEST COMBINATION
            # ---------------------------
            best_assignment = None
            best_cost = float("inf")
            
            for _, truck in valid_trucks.iterrows():
                
                for _, driver in eligible_drivers.iterrows():
                    
                    driver_id = driver["driver_id"]
                    max_hours = driver["max_hours_per_day"]
                    
                    # Get driver schedule for the day
                    day_schedule = driver_schedule[driver_id].get(date, [])
                    
                    # Overlap check
                    overlap = any(
                        not (trip_end <= s or trip_start >= e)
                        for s, e in day_schedule
                    )
                    
                    total_hours = sum(e - s for s, e in day_schedule)
                    
                    if overlap or (total_hours + (trip_end - trip_start)) > max_hours:
                        continue
                    
                    # ---------------------------
                    # COST CALCULATION
                    # ---------------------------
                    cost = compute_cost(truck, trip)
                    
                    if cost < best_cost:
                        best_cost = cost
                        best_assignment = (truck, driver_id)
            
            # ---------------------------
            # APPLY BEST ASSIGNMENT
            # ---------------------------
            if best_assignment is None:
                continue
            
            truck, assigned_driver = best_assignment
            
            # Update driver schedule
            driver_schedule[assigned_driver].setdefault(date, []).append(
                (trip_start, trip_end)
            )
            
            # Save assignment
            assignments.append({
                "assignment_id": assignment_id,
                "trip_id": trip["trip_id"],
                "truck_id": truck["truck_id"],
                "driver_id": assigned_driver,
                "route_id": trip["route_id"],
                "trip_date": date,
                "start_hour": trip_start,
                "end_hour": trip_end,
                "estimated_duration_hours": trip["estimated_duration_hours"],
                "total_cost": best_cost
            })
            
            assignment_id += 1
    
    # ---------------------------
    # OUTPUT
    # ---------------------------
    assignments_df = pd.DataFrame(assignments)
    
    return assignments_df

In [ ]:
assignments_df = assign_trips(trips_df, trucks_df, drivers_df)


Days:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/14229 [00:00<?, ?it/s]

  0%|          | 9/14229 [00:00<02:44, 86.47it/s]

  0%|          | 18/14229 [00:00<02:58, 79.80it/s]

  0%|          | 28/14229 [00:00<02:42, 87.34it/s]

  0%|          | 40/14229 [00:00<02:24, 98.42it/s]

  0%|          | 52/14229 [00:00<02:16, 103.89it/s]

  0%|          | 63/14229 [00:00<02:14, 105.03it/s]

  1%|          | 75/14229 [00:00<02:10, 108.43it/s]

  1%|          | 87/14229 [00:00<02:07, 110.57it/s]

  1%|          | 99/14229 [00:00<02:09, 108.81it/s]

  1%|          | 110/14229 [00:01<02:10, 108.21it/s]

  1%|          | 122/14229 [00:01<02:08, 109.76it/s]

  1%|          | 133/14229 [00:01<02:08, 109.34it/s]

  1%|          | 144/14229 [00:01<02:57, 79.56it/s] 

  1%|          | 154/14229 [00:01<02:53, 80.90it/s]

  1%|          | 163/14229 [00:01<02:52, 81.53it/s]

  1%|          | 175/14229 [00:01<02:35, 90.62it/s]

  1%|▏         | 185/14229 [00:01<02:32, 92.15it/s]

  1%|▏     